# Scalability of anndata x alphapepttools

Here, we demonstrate the scalability of anndata based on an example dataset by Albrecht et al, 2025

> Albrecht, V., Müller-Reif, J. B., Brennsteiner, V. & Mann, M. A Simplified Perchloric Acid Workflow With Neutralization (PCA N) for Democratizing Deep Plasma Proteomics at Population Scale. Molecular & Cellular Proteomics 24, 101071 (2025).

In this study, Albrecht et al processed more than 2000 pooled plasma samples to benchmark the reproducibility of the PCA-N workflow, as part of a larger clinical cohort study. 

In [ ]:
import alphapepttools as at
import anndata as ad
import pandas as pd
import numpy as np

In [ ]:
report_path = at.data.get_data("albrecht_pcan", output_dir=".")

./albrecht_pcan_report.tsv already exists (9664.169023513794 MB)


We showcase that anndata is capable of handling this comparatively large study and that we can extract the individual feature layers from the report:

In [ ]:
precursors: ad.AnnData = at.io.read_psm_table(
    report_path,
    search_engine="diann",
    level="precursors",
    intensity_column="Precursor.Normalised",
    feature_id_column="Precursor.Id",
    sample_id_column="Run",
    var_columns=[
        "Run",
        "Stripped.Sequence",
        "Precursor.Charge",
        "RT",
        "RT.Start",
        "RT.Stop",
        "IM",
        "Protein.Group",
        "Protein.Ids",
        "Genes",
        "MS2.Scan",
        "CScore",
        "Q.Value",
        "Precursor.Id",
        "Global.Q.Value",
        "Global.PG.Q.Value",
        "Lib.Q.Value",
        "Lib.PG.Q.Value",
    ],
)

The metadata from the PSM report is retained in the `.var` attribute of the precursor anndata object

Here, we subset the data to a all precursors whose retention time is below 5 minutes

In [ ]:
precursors[:, precursors.var_names[(precursors.var["RT"] < 5)]]  # noqa: PLR2004

View of AnnData object with n_obs × n_vars = 1801 × 3948
    var: 'Run', 'Stripped.Sequence', 'Precursor.Charge', 'RT', 'RT.Start', 'RT.Stop', 'IM', 'Protein.Group', 'Protein.Ids', 'Genes', 'MS2.Scan', 'CScore', 'Q.Value', 'PG.MaxLFQ', 'Precursor.Normalised', 'Genes.MaxLFQ', 'Global.Q.Value', 'Global.PG.Q.Value', 'Lib.Q.Value', 'Lib.PG.Q.Value'

In [ ]:
df = pd.DataFrame(precursors.X).apply(pd.to_numeric, errors="coerce")

precursors.obs_names = pd.Index(precursors.obs.index.to_numpy())
precursors.var_names = pd.Index(precursors.var.index.to_numpy())

precursors.var = pd.DataFrame(
    precursors.var.convert_dtypes(dtype_backend="numpy_nullable").values,
    index=precursors.var.index,
    columns=precursors.var.columns,
)
precursors.var["Precursor.Charge"] = precursors.var["Precursor.Charge"].astype(float)
precursors.var["RT"] = precursors.var["RT"].astype(float)
precursors.var["RT.Start"] = precursors.var["RT.Start"].astype(float)
precursors.var["RT.Stop"] = precursors.var["RT.Stop"].astype(float)
precursors.var["IM"] = precursors.var["IM"].astype(float)
precursors.var["MS2.Scan"] = precursors.var["MS2.Scan"].astype(float)
precursors.var["Q.Value"] = precursors.var["Q.Value"].astype(float)
precursors.var["Global.Q.Value"] = precursors.var["Global.Q.Value"].astype(float)
precursors.var["Global.PG.Q.Value"] = precursors.var["Global.PG.Q.Value"].astype(float)
precursors.var["Lib.Q.Value"] = precursors.var["Lib.Q.Value"].astype(float)
precursors.var["Lib.PG.Q.Value"] = precursors.var["Lib.PG.Q.Value"].astype(float)

In [ ]:
precursors.X = np.where(pd.isna(precursors.X), np.nan, precursors.X).astype(float)

In [ ]:
precursors.write_h5ad("./albrecht.precursors.h5ad")

... storing 'Run' as categorical
... storing 'Stripped.Sequence' as categorical
... storing 'Protein.Group' as categorical
... storing 'Protein.Ids' as categorical
... storing 'Genes' as categorical
... storing 'CScore' as categorical


We can similarly process protein-level and gene-level information

In [ ]:
proteins: ad.AnnData = at.io.read_psm_table(report_path, search_engine="diann", level="proteins")
proteins

AnnData object with n_obs × n_vars = 1801 × 2161

In [ ]:
genes: ad.AnnData = at.io.read_psm_table(report_path, search_engine="diann", level="genes")
genes

AnnData object with n_obs × n_vars = 1801 × 2105